In [1]:
import os
from argparse import ArgumentParser
import warnings

from omegaconf import OmegaConf
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision.utils import make_grid
from accelerate import Accelerator
from accelerate.utils import set_seed
from einops import rearrange
from tqdm import tqdm
import diffusers
from diffusers import AutoencoderKL, StableDiffusion3Pipeline

import lpips
from safetensors import safe_open
from peft import LoraConfig, get_peft_model

import math
import re

# Debugging libs: ###############
# import matplotlib.pyplot as plt
# import numpy as np
#################################

from core.utils.common import instantiate_from_config, calculate_psnr_pt, to
from core.model.mmditx import MMDiTX
from core.model.convnext import ImageConvNextDiscriminator
from core.utils.tabulate import tabulate
from core.model.other_impls import SD3Tokenizer, SDXLClipG, SDClipModel, T5XXLModel

import matplotlib.pyplot as plt

/nobackup/aryan/.conda/envs/sd35/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nobackup/aryan/.conda/envs/sd35/lib/python3.11/site-packages/gdown/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
CLIPG_CONFIG = {
    "hidden_act": "gelu",
    "hidden_size": 1280,
    "intermediate_size": 5120,
    "num_attention_heads": 20,
    "num_hidden_layers": 32,
}


class ClipG:
    def __init__(self, model_folder: str, device: str = "cpu"):
        with safe_open(
            f"{model_folder}/clip_g.safetensors", framework="pt", device="cpu"
        ) as f:
            self.model = SDXLClipG(CLIPG_CONFIG, device=device, dtype=torch.float32)
            load_into(f, self.model.transformer, "", device, torch.float32)


CLIPL_CONFIG = {
    "hidden_act": "quick_gelu",
    "hidden_size": 768,
    "intermediate_size": 3072,
    "num_attention_heads": 12,
    "num_hidden_layers": 12,
}


class ClipL:
    def __init__(self, model_folder: str):
        with safe_open(
            f"{model_folder}/clip_l.safetensors", framework="pt", device="cpu"
        ) as f:
            self.model = SDClipModel(
                layer="hidden",
                layer_idx=-2,
                device="cpu",
                dtype=torch.float32,
                layer_norm_hidden_state=False,
                return_projected_pooled=False,
                textmodel_json_config=CLIPL_CONFIG,
            )
            load_into(f, self.model.transformer, "", "cpu", torch.float32)


T5_CONFIG = {
    "d_ff": 10240,
    "d_model": 4096,
    "num_heads": 64,
    "num_layers": 24,
    "vocab_size": 32128,
}


class T5XXL:
    def __init__(self, model_folder: str, device: str = "cpu", dtype=torch.float32):
        with safe_open(
            f"{model_folder}/t5xxl.safetensors", framework="pt", device="cpu"
        ) as f:
            self.model = T5XXLModel(T5_CONFIG, device=device, dtype=dtype)
            load_into(f, self.model.transformer, "", device, dtype)


#################################################################
# Call post VAE Decode: (Will be useful for stage 2 training)
#################################################################
vae_scale_factor = 1.5305
vae_shift_factor = 0.0609

def process_in(latent): return (latent - vae_shift_factor) * vae_scale_factor
def process_out(latent): return (latent / vae_scale_factor) + vae_shift_factor
#################################################################


#################################################################
# VAE
#################################################################
from core.model.vae import SDVAE

def load_into(ckpt, model, prefix, device, dtype=None, remap=None):
    """Just a debugging-friendly hack to apply the weights in a safetensors file to the pytorch module."""
    for key in ckpt.keys():
        model_key = key
        if remap is not None and key in remap:
            model_key = remap[key]
        if model_key.startswith(prefix) and not model_key.startswith("loss."):
            path = model_key[len(prefix) :].split(".")
            obj = model
            for p in path:
                if obj is list:
                    obj = obj[int(p)]
                else:
                    obj = getattr(obj, p, None)
                    if obj is None:
                        print(
                            f"Skipping key '{model_key}' in safetensors file as '{p}' does not exist in python model"
                        )
                        break
            if obj is None:
                continue
            try:
                tensor = ckpt.get_tensor(key).to(device=device)
                if dtype is not None and tensor.dtype != torch.int32:
                    tensor = tensor.to(dtype=dtype)
                obj.requires_grad_(False)
                # print(f"K: {model_key}, O: {obj.shape} T: {tensor.shape}")
                if obj.shape != tensor.shape:
                    print(
                        f"W: shape mismatch for key {model_key}, {obj.shape} != {tensor.shape}"
                    )
                obj.set_(tensor)
            except Exception as e:
                print(f"Failed to load key '{key}' in safetensors file: {e}")
                raise e
            
CONFIGS = {
    "sd3.5_large": {
        "shift": 3.0,
        "steps": 40,
        "cfg": 4.5,
        "sampler": "dpmpp_2m",
    }
}
#################################################################

#################################################################
# Latent Space DiT
#################################################################
class ModelSamplingDiscreteFlow(torch.nn.Module):
    """Helper for sampler scheduling (ie timestep/sigma calculations) for Discrete Flow models"""

    def __init__(self, shift=1.0):
        super().__init__()
        self.shift = shift
        timesteps = 1000
        ts = self.sigma(torch.arange(1, timesteps + 1, 1))
        self.register_buffer("sigmas", ts)

    @property
    def sigma_min(self):
        return self.sigmas[0]

    @property
    def sigma_max(self):
        return self.sigmas[-1]

    def timestep(self, sigma):
        return sigma * 1000

    def sigma(self, timestep: torch.Tensor):
        timestep = timestep / 1000.0
        if self.shift == 1.0:
            return timestep
        return self.shift * timestep / (1 + (self.shift - 1) * timestep)

    def calculate_denoised(self, sigma, model_output, model_input):
        sigma = sigma.view(sigma.shape[:1] + (1,) * (model_output.ndim - 1))
        return model_input - model_output * sigma

    def noise_scaling(self, sigma, noise, latent_image, max_denoise=False):
        return sigma * noise + (1.0 - sigma) * latent_image
    
class BaseModel(torch.nn.Module):
    """Wrapper around the core MM-DiT model"""

    def __init__(
        self,
        shift=1.0,
        device=None,
        dtype=torch.float32,
        file=None,
        prefix="",
        control_model_ckpt=None,
        verbose=False,
    ):
        super().__init__()
        # Important configuration values can be quickly determined by checking shapes in the source file
        # Some of these will vary between models (eg 2B vs 8B primarily differ in their depth, but also other details change)
        patch_size = file.get_tensor(f"{prefix}x_embedder.proj.weight").shape[2]
        depth = file.get_tensor(f"{prefix}x_embedder.proj.weight").shape[0] // 64
        num_patches = file.get_tensor(f"{prefix}pos_embed").shape[1]
        pos_embed_max_size = round(math.sqrt(num_patches))
        adm_in_channels = file.get_tensor(f"{prefix}y_embedder.mlp.0.weight").shape[1]
        context_shape = file.get_tensor(f"{prefix}context_embedder.weight").shape

        qk_norm = (
            "rms"
            if f"{prefix}joint_blocks.0.context_block.attn.ln_k.weight" in file.keys()
            else None
        )
        x_block_self_attn_layers = sorted(
            [
                int(key.split(".x_block.attn2.ln_k.weight")[0].split(".")[-1])
                for key in list(
                    filter(
                        re.compile(".*.x_block.attn2.ln_k.weight").match, file.keys()
                    )
                )
            ]
        )

        context_embedder_config = {
            "target": "torch.nn.Linear",
            "params": {
                "in_features": context_shape[1],
                "out_features": context_shape[0],
            },
        }
        self.diffusion_model = MMDiTX(
            input_size=None,
            pos_embed_scaling_factor=None,
            pos_embed_offset=None,
            pos_embed_max_size=pos_embed_max_size,
            patch_size=patch_size,
            in_channels=16,
            depth=depth,
            num_patches=num_patches,
            adm_in_channels=adm_in_channels,
            context_embedder_config=context_embedder_config,
            qk_norm=qk_norm,
            x_block_self_attn_layers=x_block_self_attn_layers,
            device=device,
            dtype=dtype,
            verbose=verbose,
        )
        self.model_sampling = ModelSamplingDiscreteFlow(shift=shift)
        self.control_model = None

    def apply_model(self, x, sigma, c_crossattn=None, y=None, skip_layers=[], controlnet_cond=None):
        dtype = self.get_dtype()
        timestep = self.model_sampling.timestep(sigma).float()
        controlnet_hidden_states = None
        model_output = self.diffusion_model(
            x.to(dtype),
            timestep,
            context=c_crossattn.to(dtype),
            y=y.to(dtype),
            controlnet_hidden_states=controlnet_hidden_states,
            skip_layers=skip_layers,
        ).float()
        return self.model_sampling.calculate_denoised(sigma, model_output, x)

    def forward(self, *args, **kwargs):
        return self.apply_model(*args, **kwargs)

    def get_dtype(self):
        return self.diffusion_model.dtype
    
class SD3:
    def __init__(self, model, shift=3.0, verbose=False, device="cpu"):

        with safe_open(model, framework="pt", device="cpu") as f:
            control_model_ckpt = None
            self.model = BaseModel(
                shift=shift,
                file=f,
                prefix="model.diffusion_model.",
                device="cpu",
                dtype=torch.float16,
                control_model_ckpt=control_model_ckpt,
                verbose=verbose,
            ).eval()
            load_into(f, self.model, "model.", "cpu", torch.float16)
    
#################################################################
def fix_cond(cond):
    cond, pooled = (cond[0].half().cuda(), cond[1].half().cuda())
    return {"c_crossattn": cond, "y": pooled}


def build_text_conditioners(textenc_dir, device, dtype=torch.float32):
    tokenizer = SD3Tokenizer()
    clip_l = ClipL(textenc_dir, device=device, dtype=dtype)
    clip_g = ClipG(textenc_dir, device=device, dtype=dtype)
    t5xxl  = T5XXL(textenc_dir, device=device, dtype=dtype)
    return tokenizer, clip_l, clip_g, t5xxl


@torch.no_grad()
def encode_prompt(prompt, tokenizer, clip_l, clip_g, t5xxl, device):
    tokens = tokenizer.tokenize_with_weights(prompt)
    l_out, l_pooled = clip_l.model.encode_token_weights(tokens["l"])
    g_out, g_pooled = clip_g.model.encode_token_weights(tokens["g"])
    t5_out, t5_pooled = t5xxl.model.encode_token_weights(tokens["t5xxl"])
    lg_out = torch.cat([l_out, g_out], dim=-1)
    lg_out = F.pad(lg_out, (0, 4096 - lg_out.shape[-1]))
    cond = torch.cat([lg_out, t5_out], dim=-2).to(device)
    pooled = torch.cat([l_pooled, g_pooled], dim=-1).to(device)
    return cond.half(), pooled.half()



def prepare_sd35_inputs(pred_latent, sd3_model: SD3, cond, pooled, device, weight_dtype, model_t=200):
    B = pred_latent.size(0)
    x = pred_latent.to(device, weight_dtype)
    t = torch.full((B,), model_t, device=device, dtype=torch.float32)
    sigma = sd3_model.model.model_sampling.sigma(t)
    return dict(x=x, sigma=sigma, c_crossattn=cond, y=pooled)


In [3]:
MODEL = "/media/agarg54/Extreme SSD/sd3.5_large.safetensors" 
with safe_open(MODEL, framework="pt", device="cpu") as f:
    vae = SDVAE(device="cpu", dtype=torch.float32)
    prefix = ""
    if any(k.startswith("first_stage_model.") for k in f.keys()):
        prefix = "first_stage_model."
    load_into(f, vae, prefix, "cpu", torch.float32)
# Load qVAE from checkpoint:
vae_sd = torch.load("/nobackup1/aryan/weights/sd35_qvae_mosaic/checkpoints/0150000.pt", map_location="cpu")
vae.load_state_dict(vae_sd, strict=True)
# Freeze VAE
vae.eval().requires_grad_(False)
print(f"[+] VAE params: {sum(p.numel() for p in vae.parameters()) / 1000000:.2f}M")

[+] VAE params: 83.82M


In [40]:
# LAtent Space DiT
sd3 = SD3(MODEL, shift=3.0, device="cpu", verbose=False)
sd3.model.eval().requires_grad_(False)
# Add LoRA params to latent transformer
# target_modules = cfg.train.lora_modules
# print(f"[+] Add lora parameters to {target_modules}")
# G_lora_cfg = LoraConfig(
#     r=cfg.train.lora_rank,
#     lora_alpha=cfg.train.lora_rank, # 256
#     init_lora_weights="gaussian",
#     target_modules=target_modules,
# )
# sd3.model = get_peft_model(sd3.model, G_lora_cfg)
# lora_params = list(filter(lambda p: p.requires_grad, sd3.model.parameters()))
# assert lora_params, "Failed to find lora parameters"
# print(f"\n[+] LoRA trainable params: {sum(p.numel() for p in lora_params) / 1000000} M\n")
# for p in filter(lambda p: p.requires_grad, sd3.model.parameters()):
    # p.data = p.to(torch.float16)

# print(sd3.model)

BaseModel(
  (diffusion_model): MMDiTX(
    (x_embedder): PatchEmbed(
      (proj): Conv2d(16, 2432, kernel_size=(2, 2), stride=(2, 2))
    )
    (t_embedder): TimestepEmbedder(
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=2432, bias=True)
        (1): SiLU()
        (2): Linear(in_features=2432, out_features=2432, bias=True)
      )
    )
    (y_embedder): VectorEmbedder(
      (mlp): Sequential(
        (0): Linear(in_features=2048, out_features=2432, bias=True)
        (1): SiLU()
        (2): Linear(in_features=2432, out_features=2432, bias=True)
      )
    )
    (context_embedder): Linear(in_features=4096, out_features=2432, bias=True)
    (joint_blocks): ModuleList(
      (0-36): 37 x JointBlock(
        (context_block): DismantledBlock(
          (norm1): LayerNorm((2432,), eps=1e-06, elementwise_affine=False)
          (attn): SelfAttention(
            (qkv): Linear(in_features=2432, out_features=7296, bias=True)
            (proj): Linear(in_fea

In [41]:
print(f"[+] Latent DiT params: {sum(p.numel() for p in sd3.model.parameters()) / 1_000_000_000:.2f}B")

[+] Latent DiT params: 8.06B


In [25]:
def list_leaf_modules(m):
    for name, mod in m.named_modules():
        # leaf-ish linear/conv
        if isinstance(mod, (torch.nn.Linear, torch.nn.Conv2d)):
            print(name)

In [ ]:
# list_leaf_modules(sd3.model)

diffusion_model.x_embedder.proj
diffusion_model.t_embedder.mlp.0
diffusion_model.t_embedder.mlp.2
diffusion_model.y_embedder.mlp.0
diffusion_model.y_embedder.mlp.2
diffusion_model.context_embedder
diffusion_model.joint_blocks.0.context_block.attn.qkv
diffusion_model.joint_blocks.0.context_block.attn.proj
diffusion_model.joint_blocks.0.context_block.mlp.fc1
diffusion_model.joint_blocks.0.context_block.mlp.fc2
diffusion_model.joint_blocks.0.context_block.adaLN_modulation.1
diffusion_model.joint_blocks.0.x_block.attn.qkv
diffusion_model.joint_blocks.0.x_block.attn.proj
diffusion_model.joint_blocks.0.x_block.mlp.fc1
diffusion_model.joint_blocks.0.x_block.mlp.fc2
diffusion_model.joint_blocks.0.x_block.adaLN_modulation.1
diffusion_model.joint_blocks.1.context_block.attn.qkv
diffusion_model.joint_blocks.1.context_block.attn.proj
diffusion_model.joint_blocks.1.context_block.mlp.fc1
diffusion_model.joint_blocks.1.context_block.mlp.fc2
diffusion_model.joint_blocks.1.context_block.adaLN_modulatio

In [42]:
TARGETS = [
  "x_embedder.proj",
  "t_embedder.mlp.0",
  "t_embedder.mlp.2",
  "y_embedder.mlp.0",
  "y_embedder.mlp.2",
  "context_embedder",
  "context_block.attn.qkv",
  "context_block.attn.proj",
  "x_block.attn.qkv",
  "x_block.attn.proj",
  "context_block.mlp.fc1",
  "context_block.mlp.fc2",
  "x_block.mlp.fc1",
  "x_block.mlp.fc2",
  "final_layer.linear",
  # optional (conditioning-strength adaptation):
  # "context_block.adaLN_modulation.1",
  # "x_block.adaLN_modulation.1",
  # "final_layer.adaLN_modulation.1",
]

In [43]:
def attach_lora_and_report(base_model, r=16, alpha=16, dropout=0.0):
    cfg = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=TARGETS,
        init_lora_weights="gaussian",
    )
    peft_model = get_peft_model(base_model, cfg)

    # 1) Collect which named modules got LoRA adapters
    lora_param_names = [n for n, p in peft_model.named_parameters() if "lora_" in n]
    touched_modules = sorted(set(n.split(".lora_")[0] for n in lora_param_names))

    print(f"[LoRA] #adapter tensors: {len(lora_param_names)}")
    print(f"[LoRA] #unique modules with adapters: {len(touched_modules)}")
    for m in touched_modules[:30]:
        print("  -", m)
    if len(touched_modules) > 30:
        print("  ... (truncated) ...")

    # 2) Count learnable vs total params
    tot = sum(p.numel() for _, p in peft_model.named_parameters())
    learn = sum(p.numel() for _, p in peft_model.named_parameters() if p.requires_grad)
    print(f"[Params] total={tot/1e6:.2f}M | learnable={learn/1e6:.2f}M")

    # 3) Sanity: ensure at least one adapter in each *class* of targets
    classes = {
        "x_embedder": "diffusion_model.x_embedder.proj",
        "t_embedder": "diffusion_model.t_embedder.mlp",
        "y_embedder": "diffusion_model.y_embedder.mlp",
        "ctx_attn": ".context_block.attn.",
        "x_attn": ".x_block.attn.",
        "ctx_mlp": ".context_block.mlp.",
        "x_mlp": ".x_block.mlp.",
        "final": "diffusion_model.final_layer.linear",
    }
    missing = []
    for tag, key in classes.items():
        if not any(key in m for m in touched_modules):
            missing.append(tag)
    if missing:
        print("[WARN] No LoRA attached for classes:", missing)
    else:
        print("[OK] LoRA attached for all target classes.")

    # 4) Check gradients: only LoRA params should require_grad=True
    bad = [n for n, p in peft_model.named_parameters() if p.requires_grad and "lora_" not in n]
    if bad:
        print("[WARN] Non-LoRA params are trainable:", bad[:20], "...")
    else:
        print("[OK] Only LoRA params are trainable.")

    return peft_model

In [44]:
blyat_1 = attach_lora_and_report(sd3.model, r=8, alpha=8, dropout=0.0)

[LoRA] #adapter tensors: 616
[LoRA] #unique modules with adapters: 308
  - base_model.model.diffusion_model.context_embedder
  - base_model.model.diffusion_model.final_layer.linear
  - base_model.model.diffusion_model.joint_blocks.0.context_block.attn.proj
  - base_model.model.diffusion_model.joint_blocks.0.context_block.attn.qkv
  - base_model.model.diffusion_model.joint_blocks.0.context_block.mlp.fc1
  - base_model.model.diffusion_model.joint_blocks.0.context_block.mlp.fc2
  - base_model.model.diffusion_model.joint_blocks.0.x_block.attn.proj
  - base_model.model.diffusion_model.joint_blocks.0.x_block.attn.qkv
  - base_model.model.diffusion_model.joint_blocks.0.x_block.mlp.fc1
  - base_model.model.diffusion_model.joint_blocks.0.x_block.mlp.fc2
  - base_model.model.diffusion_model.joint_blocks.1.context_block.attn.proj
  - base_model.model.diffusion_model.joint_blocks.1.context_block.attn.qkv
  - base_model.model.diffusion_model.joint_blocks.1.context_block.mlp.fc1
  - base_model.model